In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_splitimport pandas as pd

In [ ]:
# Preset values

years = [2025, 2026]
strt_month = 5

res_prop = "Residential"
res_sub = "SingleFamilyResidence"

crit_cols = ["ListPrice", "ListingKey", "ListingContractDate", "PurchaseContractDate",
             "CloseDate", "ClosePrice", "Latitude", "Longitude", "PropertyType", "BedroomsTotal",
             "BathroomsTotalInteger", "LivingArea", "DaysOnMarket", "UnparsedAddress"]                # Define list of critical columns

sub_cols = ["ListPrice", "ListingKey", "ClosePrice", "Latitude", "Longitude", "PropertyType",
             "BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "DaysOnMarket"]                  # Define subset columns

In [ ]:
def load_data(yrs=years, strt=strt_month):
  """
  Takes an inputted list of years and integer-valued start month for the CRMLS files
  Returns merged DataFrame of read csvs
  """
  main_df = pd.DataFrame()                                              # Defines blank DataFrame
  months = [str(i) if i > 9 else "0"+str(i) for i in range (1,13)]      # Defines proper format and type of month values

  for a in range(strt-1, 12):           # Iterates through months, starting with inputted strt month
    try:
      df = pd.read_csv("CRMLSSold"+str(yrs[0])+months[a]+".csv", low_memory=False)    # Reads csv
      main_df = pd.concat([main_df, df], ignore_index=True)                           # Appends DataFrame of csv to "main_df"
    except:
      df = pd.read_csv("CRMLSSold"+str(yrs[0])+months[a]+"_filled.csv", low_memory=False)
      main_df = pd.concat([main_df, df], ignore_index=True)

  if len(yrs) > 2:                    # If "yrs" list is greater than two,
    for b in range(1,len(yrs)-1):       # For each year in "yrs", except the first and last
      for c in months:                    # Iterates through each month in "months" (whole year for yrs[b])
        try:
          df = pd.read_csv("CRMLSSold"+str(yrs[b])+c+".csv", low_memory=False)        # Reads csv
          main_df = pd.concat([main_df, df], ignore_index=True)                       # Appends DataFrame of csv to "main_df"
        except:
          df = pd.read_csv("CRMLSSold"+str(yrs[b])+c+"_filled.csv", low_memory=False)
          main_df = pd.concat([main_df, df], ignore_index=True)

  elif len(yrs) == 2:           # Elif,
    for d in range(0, strt):      # Iterates through months upto the strt for the last year in "yrs"
      try:
        df = pd.read_csv("CRMLSSold"+str(yrs[len(yrs)-1])+months[d]+".csv", low_memory=False)   # Reads csv
        main_df = pd.concat([main_df, df], ignore_index=True)                                   # Appends DataFrame of csv to "main_df"
      except:
        df = pd.read_csv("CRMLSSold"+str(yrs[len(yrs)-1])+months[d]+"_filled.csv", low_memory=False)
        main_df = pd.concat([main_df, df], ignore_index=True)

  return main_df

In [ ]:
def get_subset(df, prop=res_prop, sub_prop=res_sub, cols=sub_cols):
  """
  Takes the DataFrame, PropertyType value (str), PropertySubType value (str), and list of columns for subset DataFrame
  Returns subset of inputted DataFrame, based on inputted parameters
  """
  df = df[(df["PropertyType"] == prop)&(df["PropertySubType"] == sub_prop)]     # Takes subset of DataFrame based on property type and subtype
  sub_df = df[cols].reset_index(drop=True)                                      # Takes subset of columns for DataFrame
  return sub_df

In [ ]:
def clean_data(df, crit=sub_cols):
  """
  Takes a DataFrame and list of critical columns
  Returns a DataFrame with duplicate values dropped, as well as null and zeros based on subsets of columns
  """
  clean_df = df.dropna(subset=crit)
  clean_df = df.drop_duplicates()
  clean_df = df[(df["ClosePrice"]>0) & (df["LivingArea"]>0) & (df["BathroomsTotalInteger"]>0) & (df["DaysOnMarket"]>0)]
    # ListPrice, Latitude/Longitude, BedroomsTotal
  return clean_df.reset_index(drop=True)

In [ ]:
def encode_and_split(df):
  """
  Takes a DataFrame
  Encodes "PropertyType" column
  Returns a defined training and test set for the DataFrame
  """
  re_df = pd.get_dummies(df, columns=["PropertyType"])    # Encoded "PropertyType" column of the df

  mo_yr = [a[0:7] for a in df["CloseDate"].to_list()]
  te_set = [b for b in range(len(mo_yr)) if mo_yr[b] == "2026-05"]
  te_rng = te_set[0::len(te_set)-1]
  x_tr, x_te = re_df[0:te_rng[0]], re_df[te_rng[0]:te_rng[1]]
  return x_tr, x_te

In [ ]:
def main():
  main = load_data()
  sub = get_subset(main)
  clean = clean_data(sub)
  tr, te = encode_and_split(clean)
  return tr, te

In [ ]:
if __name__ == "__main__":
  main()